### Swin Video Transformer

#### quick test mode conversion

In [63]:
# import os
# import cv2
# import pandas as pd
# import decord
# from tqdm import tqdm
# import numpy as np
# import matplotlib.pyplot as plt
# from IPython.display import Video, display

# # ============================================================
# # CONFIGURATION (FAST MODE WITH MARGIN)
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/timesformer-dataset-train"
# OUTPUT_ROOT = "/home1/jtt_1/mtp/swin-person-dataset-quicktest"
# CSV_PATH = os.path.join(ROOT, "labels.csv")

# MAX_VIDEOS = 2
# MAX_TRACKS = 5
# FRAMES_PER_CLIP = 8
# FRAME_SKIP = 1
# MIN_CLIP_LEN = 2
# MARGIN_RATIO = 0.2
# DEBUG = True
# FPS = 10

# FRAME_WIDTH, FRAME_HEIGHT = 1280, 720  # to clip expanded coords safely

# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # LOAD LABEL SUMMARY
# # ============================================================
# df = pd.read_csv(CSV_PATH)
# print(f"✅ Loaded {len(df)} video entries from labels.csv")

# df = df.sample(min(MAX_VIDEOS, len(df))).reset_index(drop=True)
# print(f"⚡ Running quick test on {len(df)} videos")

# # ============================================================
# # QUICK CLIP CONVERSION WITH MARGINS
# # ============================================================
# decord.bridge.set_bridge("torch")
# summary_rows = []

# for _, row in tqdm(df.iterrows(), total=len(df), desc="Quick converting"):
#     video_path = row["VideoPath"]
#     bbox_path = row["BBoxCSVPath"]
#     video_id = os.path.splitext(os.path.basename(video_path))[0]

#     if not (os.path.exists(video_path) and os.path.exists(bbox_path)):
#         print(f"⚠️ Missing {video_path} or bbox file.")
#         continue

#     bbox_df = pd.read_csv(bbox_path)
#     if not {"frame", "track_id", "xmin", "ymin", "xmax", "ymax", "actions"}.issubset(bbox_df.columns):
#         print(f"⚠️ Malformed bbox file for {video_id}")
#         continue

#     # load video frames
#     try:
#         vr = decord.VideoReader(video_path)
#     except Exception as e:
#         print(f"❌ Error reading {video_path}: {e}")
#         continue

#     # iterate few persons only
#     for track_id, person_df in bbox_df.groupby("track_id"):
#         if track_id > MAX_TRACKS:
#             break
#         person_df = person_df.sort_values("frame")

#         frames = person_df["frame"].values
        

#         if len(frames) < MIN_CLIP_LEN:
#             if DEBUG:
#                 print(f"   ⚠️ Track {track_id} too short ({len(frames)} frames)")
#             continue


#         # take only first short clip for each track
#         start_f = frames[0]
#         end_f = min(frames[0] + FRAMES_PER_CLIP, frames[-1])
#         subset = person_df[(person_df["frame"] >= start_f) & (person_df["frame"] <= end_f)]

#         actions = subset["actions"].dropna().unique().tolist()
#         if DEBUG and not actions:
#             print(f"   ⚠️ No valid actions for track {track_id} ({video_id})")
#         actions_str = ",".join(actions)

#         clip_name = f"{video_id}_track{track_id}_f{start_f}-{end_f}.mp4"
#         out_path = os.path.join(OUTPUT_ROOT, clip_name)

#         writer = None
#         clip_frames = 0
#         for _, det in subset.iloc[::FRAME_SKIP].iterrows():
#             f_idx = int(det["frame"])
#             xmin, ymin, xmax, ymax = map(float, [det["xmin"], det["ymin"], det["xmax"], det["ymax"]])

#             # ===== APPLY MARGIN =====
#             width = xmax - xmin
#             height = ymax - ymin
#             xmin -= width * MARGIN_RATIO
#             xmax += width * MARGIN_RATIO
#             ymin -= height * MARGIN_RATIO
#             ymax += height * MARGIN_RATIO

#             # Clip to valid frame bounds
#             xmin, ymin = max(0, int(xmin)), max(0, int(ymin))
#             xmax, ymax = min(int(xmax), FRAME_WIDTH), min(int(ymax), FRAME_HEIGHT)

#             try:
#                 frame = vr[f_idx].numpy()
#             except:
#                 continue

#             crop = frame[ymin:ymax, xmin:xmax]
#             if crop.size == 0:
#                 continue

#             if writer is None:
#                 h, w, _ = crop.shape
#                 writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))

#             crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
#             writer.write(crop_bgr)
#             clip_frames += 1

#         if writer:
#             writer.release()

#         if clip_frames >= MIN_CLIP_LEN:
#             summary_rows.append([
#                 video_id, track_id, start_f, end_f, clip_frames, actions_str, out_path
#             ])

# # ============================================================
# # SAVE QUICK SUMMARY
# # ============================================================
# summary_df = pd.DataFrame(summary_rows, columns=[
#     "VideoID", "TrackID", "StartFrame", "EndFrame", "FrameCount", "Actions", "ClipPath"
# ])
# out_csv = os.path.join(OUTPUT_ROOT, "person_labels_quick_margin.csv")
# summary_df.to_csv(out_csv, index=False)

# print(f"\n✅ Created {len(summary_df)} quick test clips (with margin)")
# print(f"🧾 Summary CSV saved at: {out_csv}")
# print(f"📁 Clips saved to: {OUTPUT_ROOT}")

# # ============================================================
# # VISUALIZATION
# # ============================================================
# print("\n🎞️ Visualizing a few sample clips with margins...")

# VIS_SAVE_DIR = os.path.join(OUTPUT_ROOT, "visual_check")
# os.makedirs(VIS_SAVE_DIR, exist_ok=True)

# for _, row in summary_df.sample(min(3, len(summary_df))).iterrows():
#     clip_path = row["ClipPath"]
#     actions = row["Actions"]
#     track_id = row["TrackID"]
#     video_id = row["VideoID"]

#     print(f"\n🎥 Clip: {os.path.basename(clip_path)}")
#     print(f"   VideoID: {video_id}, TrackID: {track_id}")
#     print(f"   Actions: {actions}")

#     cap = cv2.VideoCapture(clip_path)
#     frames = []
#     count = 0
#     while True:
#         ret, frame = cap.read()
#         if not ret or count > 5:
#             break
#         frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         frames.append(frame_rgb)
#         count += 1
#     cap.release()

#     if len(frames) == 0:
#         print("   ⚠️ No frames read from clip.")
#         continue

#     first_frame_path = os.path.join(VIS_SAVE_DIR, f"{os.path.basename(clip_path)}_f0.jpg")
#     cv2.imwrite(first_frame_path, cv2.cvtColor(frames[0], cv2.COLOR_RGB2BGR))
#     print(f"   ✅ Saved first frame as {first_frame_path}")

#     # Robust visualization even if only one frame
#     fig, axes = plt.subplots(1, len(frames), figsize=(15, 5))

#     # If there's only one frame, axes is not iterable
#     if len(frames) == 1:
#         axes = [axes]

#     for i, f in enumerate(frames):
#         axes[i].imshow(f)
#         axes[i].axis('off')
#         axes[i].set_title(f'Frame {i}')
#     plt.suptitle(f'Actions: {actions}', fontsize=14)
#     plt.show()


#     try:
#         display(Video(clip_path, embed=True, width=400))
#     except:
#         print(f"   ▶️ Video: {clip_path}")

# print("\n✅ Quick conversion + visualization (with margin) complete!")
# print(f"🧩 Output directory: {OUTPUT_ROOT}")


#### train Conversion mode

In [64]:
# # ============================================================
# # build_person_clips_train_optimized.py
# # ============================================================
# import os
# import cv2
# import pandas as pd
# import decord
# from tqdm import tqdm
# import numpy as np
# import torch

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/timesformer-dataset-train"
# OUTPUT_ROOT = "/home1/jtt_1/mtp/swin-person-dataset-full"
# CSV_PATH = os.path.join(ROOT, "labels.csv")

# # Clip parameters
# FRAMES_PER_CLIP = 16         # number of frames per clip
# FRAME_SKIP = 2               # skip every 2nd frame for temporal spacing
# MIN_CLIP_LEN = 8             # skip too-short tracks
# MARGIN_RATIO = 0.15          # expand bbox by 15% for context
# STEP = FRAMES_PER_CLIP // 2  # 50% overlap for smoother coverage
# DEFAULT_FPS = 10             # fallback FPS if not found

# # Ensure output directory
# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # LOAD LABEL SUMMARY
# # ============================================================
# df = pd.read_csv(CSV_PATH)
# print(f"✅ Loaded {len(df)} videos from labels.csv")

# # ============================================================
# # DECORD SETUP
# # ============================================================
# decord.bridge.set_bridge("torch")
# summary_rows = []

# print(f"\n🚀 Starting full dataset conversion ({len(df)} videos)...\n")

# # ============================================================
# # PROCESS EACH VIDEO
# # ============================================================
# for _, row in tqdm(df.iterrows(), total=len(df), desc="Converting all videos"):
#     video_path = row["VideoPath"]
#     bbox_path = row["BBoxCSVPath"]
#     video_id = os.path.splitext(os.path.basename(video_path))[0]

#     # --- File checks ---
#     if not (os.path.exists(video_path) and os.path.exists(bbox_path)):
#         print(f"⚠️ Skipping {video_id}: Missing video or bbox file.")
#         continue

#     bbox_df = pd.read_csv(bbox_path)
#     required_cols = {"frame", "track_id", "xmin", "ymin", "xmax", "ymax", "actions"}
#     if not required_cols.issubset(bbox_df.columns):
#         print(f"⚠️ Skipping {video_id}: Malformed bbox CSV.")
#         continue

#     # --- Load video frames ---
#     try:
#         vr = decord.VideoReader(video_path)
#         total_frames = len(vr)
#         FRAME_HEIGHT, FRAME_WIDTH = vr[0].shape[:2]
#         try:
#             video_fps = vr.get_avg_fps()
#             if not np.isfinite(video_fps) or video_fps <= 0:
#                 video_fps = DEFAULT_FPS
#         except Exception:
#             video_fps = DEFAULT_FPS
#     except Exception as e:
#         print(f"❌ Error reading {video_id}: {e}")
#         continue

#     # --- Process each tracked person ---
#     for track_id, person_df in bbox_df.groupby("track_id"):
#         person_df = person_df.sort_values("frame")
#         frames = person_df["frame"].values

#         if len(frames) < MIN_CLIP_LEN:
#             continue

#         # Sliding window with overlap
#         for i in range(0, len(frames) - FRAMES_PER_CLIP, STEP):
#             start_f = frames[i]
#             end_f = frames[i + FRAMES_PER_CLIP - 1]
#             subset = person_df[(person_df["frame"] >= start_f) & (person_df["frame"] <= end_f)]

#             if subset.empty:
#                 continue

#             # Multi-label actions
#             actions = subset["actions"].dropna().unique().tolist()
#             if not actions:
#                 continue
#             actions_str = ",".join(actions)

#             clip_name = f"{video_id}_track{track_id}_f{start_f}-{end_f}.mp4"
#             out_path = os.path.join(OUTPUT_ROOT, clip_name)
#             rel_path = os.path.relpath(out_path, OUTPUT_ROOT)

#             writer = None
#             clip_frames = 0

#             for _, det in subset.iloc[::FRAME_SKIP].iterrows():
#                 f_idx = int(det["frame"])
#                 if f_idx >= total_frames:
#                     continue

#                 xmin, ymin, xmax, ymax = map(float, [det["xmin"], det["ymin"], det["xmax"], det["ymax"]])

#                 # === Apply margin ===
#                 width = xmax - xmin
#                 height = ymax - ymin
#                 xmin -= width * MARGIN_RATIO
#                 xmax += width * MARGIN_RATIO
#                 ymin -= height * MARGIN_RATIO
#                 ymax += height * MARGIN_RATIO

#                 # Clip to valid bounds
#                 xmin, ymin = max(0, int(xmin)), max(0, int(ymin))
#                 xmax, ymax = min(int(xmax), FRAME_WIDTH), min(int(ymax), FRAME_HEIGHT)

#                 try:
#                     frame = vr[f_idx].numpy()
#                 except Exception:
#                     continue

#                 crop = frame[ymin:ymax, xmin:xmax]
#                 if crop.size == 0:
#                     continue

#                 if writer is None:
#                     h, w, _ = crop.shape
#                     writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), video_fps, (w, h))

#                 crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
#                 writer.write(crop_bgr)
#                 clip_frames += 1

#             if writer:
#                 writer.release()

#             if clip_frames >= MIN_CLIP_LEN:
#                 summary_rows.append([
#                     video_id, track_id, start_f, end_f, clip_frames, actions_str, rel_path, video_fps
#                 ])

#     # --- Cleanup ---
#     del vr
#     torch.cuda.empty_cache()

# # ============================================================
# # SAVE SUMMARY CSV
# # ============================================================
# summary_df = pd.DataFrame(summary_rows, columns=[
#     "VideoID", "TrackID", "StartFrame", "EndFrame", "FrameCount",
#     "Actions", "ClipPath", "FPS"
# ])
# out_csv = os.path.join(OUTPUT_ROOT, "person_labels.csv")
# summary_df.to_csv(out_csv, index=False)

# print(f"\n✅ Conversion complete: {len(summary_df)} person clips created")
# print(f"📁 Clips saved in: {OUTPUT_ROOT}")
# print(f"🧾 Summary CSV: {out_csv}")

# # ============================================================
# # DATASET STATS
# # ============================================================
# unique_actions = set()
# for acts in summary_df["Actions"].dropna():
#     for a in str(acts).split(","):
#         a = a.strip()
#         if a:
#             unique_actions.add(a)

# print(f"\n🏷️ Unique actions found: {len(unique_actions)}")
# print(f"➡️ {sorted(unique_actions)}")


#### Test set datsets

In [65]:
# # ============================================================
# # build_person_clips_full_optimized.py
# # ============================================================
# import os
# import cv2
# import pandas as pd
# import decord
# from tqdm import tqdm
# import numpy as np
# import torch

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/timesformer-dataset-test"
# OUTPUT_ROOT = "/home1/jtt_1/mtp/swin-person-dataset-full-test"
# CSV_PATH = os.path.join(ROOT, "labels.csv")

# # Clip parameters
# FRAMES_PER_CLIP = 16        # number of frames per clip
# FRAME_SKIP = 2              # skip every 2nd frame for temporal spacing
# MIN_CLIP_LEN = 8            # skip tracks shorter than this
# MARGIN_RATIO = 0.15         # expand bbox by 15% for context
# STEP = FRAMES_PER_CLIP // 2 # 50% overlap
# DEFAULT_FPS = 10            # fallback FPS if not available

# # Ensure output directory
# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # LOAD LABEL SUMMARY
# # ============================================================
# df = pd.read_csv(CSV_PATH)
# print(f"✅ Loaded {len(df)} videos from labels.csv")

# # ============================================================
# # DECORD SETUP
# # ============================================================
# decord.bridge.set_bridge("torch")
# summary_rows = []

# print(f"\n🚀 Starting full dataset conversion ({len(df)} videos)...\n")

# # ============================================================
# # PROCESS EACH VIDEO
# # ============================================================
# for _, row in tqdm(df.iterrows(), total=len(df), desc="Converting all videos"):
#     video_path = row["VideoPath"]
#     bbox_path = row["BBoxCSVPath"]
#     video_id = os.path.splitext(os.path.basename(video_path))[0]

#     # --- File checks ---
#     if not (os.path.exists(video_path) and os.path.exists(bbox_path)):
#         print(f"⚠️ Skipping {video_id}: Missing video or bbox file.")
#         continue

#     bbox_df = pd.read_csv(bbox_path)
#     required_cols = {"frame", "track_id", "xmin", "ymin", "xmax", "ymax", "actions"}
#     if not required_cols.issubset(bbox_df.columns):
#         print(f"⚠️ Skipping {video_id}: Malformed bbox CSV.")
#         continue

#     # --- Load video frames ---
#     try:
#         vr = decord.VideoReader(video_path)
#         total_frames = len(vr)
#         FRAME_HEIGHT, FRAME_WIDTH = vr[0].shape[:2]
#         try:
#             video_fps = vr.get_avg_fps()
#             if not np.isfinite(video_fps) or video_fps <= 0:
#                 video_fps = DEFAULT_FPS
#         except Exception:
#             video_fps = DEFAULT_FPS
#     except Exception as e:
#         print(f"❌ Error reading {video_id}: {e}")
#         continue

#     # --- Process each tracked person ---
#     for track_id, person_df in bbox_df.groupby("track_id"):
#         person_df = person_df.sort_values("frame")
#         frames = person_df["frame"].values

#         if len(frames) < MIN_CLIP_LEN:
#             continue

#         # Slide window (with overlap)
#         for i in range(0, len(frames) - FRAMES_PER_CLIP, STEP):
#             start_f = frames[i]
#             end_f = frames[i + FRAMES_PER_CLIP - 1]
#             subset = person_df[(person_df["frame"] >= start_f) & (person_df["frame"] <= end_f)]

#             if subset.empty:
#                 continue

#             # Collect multi-label actions
#             actions = subset["actions"].dropna().unique().tolist()
#             if not actions:
#                 continue
#             actions_str = ",".join(actions)

#             clip_name = f"{video_id}_track{track_id}_f{start_f}-{end_f}.mp4"
#             out_path = os.path.join(OUTPUT_ROOT, clip_name)
#             rel_path = os.path.relpath(out_path, OUTPUT_ROOT)

#             writer = None
#             clip_frames = 0

#             for _, det in subset.iloc[::FRAME_SKIP].iterrows():
#                 f_idx = int(det["frame"])
#                 if f_idx >= total_frames:
#                     continue

#                 xmin, ymin, xmax, ymax = map(float, [det["xmin"], det["ymin"], det["xmax"], det["ymax"]])

#                 # === Apply margin ===
#                 width = xmax - xmin
#                 height = ymax - ymin
#                 xmin -= width * MARGIN_RATIO
#                 xmax += width * MARGIN_RATIO
#                 ymin -= height * MARGIN_RATIO
#                 ymax += height * MARGIN_RATIO

#                 # Clip to valid bounds
#                 xmin, ymin = max(0, int(xmin)), max(0, int(ymin))
#                 xmax, ymax = min(int(xmax), FRAME_WIDTH), min(int(ymax), FRAME_HEIGHT)

#                 try:
#                     frame = vr[f_idx].numpy()
#                 except Exception:
#                     continue

#                 crop = frame[ymin:ymax, xmin:xmax]
#                 if crop.size == 0:
#                     continue

#                 if writer is None:
#                     h, w, _ = crop.shape
#                     writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), video_fps, (w, h))

#                 crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
#                 writer.write(crop_bgr)
#                 clip_frames += 1

#             if writer:
#                 writer.release()

#             if clip_frames >= MIN_CLIP_LEN:
#                 summary_rows.append([
#                     video_id, track_id, start_f, end_f, clip_frames, actions_str, rel_path, video_fps
#                 ])

#     # --- Clean memory ---
#     del vr
#     torch.cuda.empty_cache()

# # ============================================================
# # SAVE SUMMARY CSV
# # ============================================================
# summary_df = pd.DataFrame(summary_rows, columns=[
#     "VideoID", "TrackID", "StartFrame", "EndFrame", "FrameCount",
#     "Actions", "ClipPath", "FPS"
# ])
# out_csv = os.path.join(OUTPUT_ROOT, "person_labels.csv")
# summary_df.to_csv(out_csv, index=False)

# print(f"\n✅ Conversion complete: {len(summary_df)} person clips created")
# print(f"📁 Clips saved in: {OUTPUT_ROOT}")
# print(f"🧾 Summary CSV: {out_csv}")

# # ============================================================
# # DATASET STATS
# # ============================================================
# unique_actions = set()
# for acts in summary_df["Actions"].dropna():
#     for a in str(acts).split(","):
#         a = a.strip()
#         if a:
#             unique_actions.add(a)

# print(f"\n🏷️ Unique actions found: {len(unique_actions)}")
# print(f"➡️ {sorted(unique_actions)}")


#### Visulization

In [66]:
# # ============================================================
# # visualize_person_clips_random_frames.py
# # ============================================================
# import os
# import cv2
# import pandas as pd
# import matplotlib.pyplot as plt
# import random

# # ============================================================
# # CONFIGURATION
# # ============================================================
# DATASET_ROOT = "/home1/jtt_1/mtp/swin-person-dataset-full-test"
# CSV_PATH = os.path.join(DATASET_ROOT, "person_labels.csv")

# NUM_SAMPLES = 4        # number of clips to visualize
# MAX_FRAMES = 6         # how many frames to show per clip
# RANDOM_SEED = None     # None => random each run

# if RANDOM_SEED is not None:
#     random.seed(RANDOM_SEED)

# # ============================================================
# # LOAD LABEL CSV
# # ============================================================
# if not os.path.exists(CSV_PATH):
#     raise FileNotFoundError(f"❌ CSV not found: {CSV_PATH}")

# df = pd.read_csv(CSV_PATH)
# print(f"✅ Loaded {len(df)} clips from {CSV_PATH}")

# # Extract unique actions
# all_actions = set()
# for acts in df["Actions"].dropna():
#     for a in str(acts).split(","):
#         all_actions.add(a.strip())
# print(f"🏷️ Unique action labels: {sorted(list(all_actions))}")

# # Random sample (different each run)
# sampled = df.sample(min(NUM_SAMPLES, len(df)))
# print(f"\n🎞️ Randomly selected {len(sampled)} clips for visualization...\n")

# # ============================================================
# # VISUALIZATION LOOP
# # ============================================================
# for i, (_, row) in enumerate(sampled.iterrows(), start=1):
#     clip_path = str(row["ClipPath"]).strip()
#     if not os.path.isabs(clip_path):
#         clip_path = os.path.join(DATASET_ROOT, clip_path)
#     clip_path = os.path.normpath(clip_path)

#     video_id = row.get("VideoID", "unknown")
#     track_id = row.get("TrackID", "N/A")
#     actions = str(row["Actions"])

#     exists = "✅ Exists" if os.path.exists(clip_path) else "❌ Missing"

#     print(f"🎥 [{i}] {os.path.basename(clip_path)}")
#     print(f"   📽️ VideoID: {video_id} | TrackID: {track_id}")
#     print(f"   🏷️ Actions: {actions}")
#     print(f"   📁 Path: {clip_path}")
#     print(f"   🟩 Status: {exists}")

#     if not os.path.exists(clip_path):
#         print("   ⚠️ Skipping missing clip.\n")
#         continue

#     # --- Load frames ---
#     cap = cv2.VideoCapture(clip_path)
#     frames = []
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     sample_indices = sorted(random.sample(range(total_frames), min(MAX_FRAMES, total_frames)))
    
#     for idx in sample_indices:
#         cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#         ret, frame = cap.read()
#         if not ret:
#             continue
#         frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         frames.append((idx, frame_rgb))
#     cap.release()

#     if len(frames) == 0:
#         print("   ⚠️ No frames could be read.\n")
#         continue

#     # --- Display sampled frames ---
#     fig, axes = plt.subplots(1, len(frames), figsize=(15, 4))
#     if len(frames) == 1:
#         axes = [axes]
#     for j, (idx, frame) in enumerate(frames):
#         axes[j].imshow(frame)
#         axes[j].axis('off')
#         axes[j].set_title(f"Frame {idx}", fontsize=9)
#     plt.suptitle(f"Clip {i}: {actions}", fontsize=14, color='green', fontweight='bold')
#     plt.tight_layout()
#     plt.show()

# print("\n✅ Visualization complete (no files saved, random clips shown each run).")


#### Split Dataset

In [67]:
import os

# Folder path
DATASET_DIR = "/home1/jtt_1/mtp/swin-person-dataset-full"

# Files to delete
FILES_TO_DELETE = [
    "train_list.txt",
    "train_manifest.csv",
    "train_manifest_filtered.csv",
    "val_list.txt",
    "val_manifest.csv",
    "val_manifest_filtered.csv"
]

# Delete safely
for fname in FILES_TO_DELETE:
    fpath = os.path.join(DATASET_DIR, fname)
    if os.path.exists(fpath):
        try:
            os.remove(fpath)
            print(f"🗑️ Deleted: {fname}")
        except Exception as e:
            print(f"⚠️ Error deleting {fname}: {e}")
    else:
        print(f"⚠️ Not found: {fname}")

print("\n✅ Cleanup complete.")


🗑️ Deleted: train_list.txt
🗑️ Deleted: train_manifest.csv
🗑️ Deleted: train_manifest_filtered.csv
🗑️ Deleted: val_list.txt
🗑️ Deleted: val_manifest.csv
🗑️ Deleted: val_manifest_filtered.csv

✅ Cleanup complete.


In [68]:
# ============================================================
# build_train_val_split.py
# ============================================================
import os
import pandas as pd
import random
from collections import Counter
from sklearn.model_selection import train_test_split

# ============================================================
# CONFIGURATION
# ============================================================
DATASET_ROOT = "/home1/jtt_1/mtp/swin-person-dataset-full"
CSV_PATH = os.path.join(DATASET_ROOT, "person_labels.csv")
TRAIN_LIST = os.path.join(DATASET_ROOT, "train_list.txt")
VAL_LIST   = os.path.join(DATASET_ROOT, "val_list.txt")

SPLIT_RATIO = 0.8   # 80% train / 20% val
SEED = 42
random.seed(SEED)

# ============================================================
# LOAD DATA
# ============================================================
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ person_labels.csv not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"✅ Loaded {len(df)} clips from {CSV_PATH}")

# Clean and parse labels
df["Actions"] = df["Actions"].fillna("").astype(str)
df["labels_split"] = df["Actions"].apply(lambda x: [a.strip() for a in x.split(",") if a.strip()])

# ============================================================
# SPLIT BY VIDEOID (to prevent leakage)
# ============================================================
unique_videos = sorted(df["VideoID"].unique())
train_vids, val_vids = train_test_split(
    unique_videos, test_size=(1 - SPLIT_RATIO), random_state=SEED
)

train_df = df[df["VideoID"].isin(train_vids)].copy()
val_df   = df[df["VideoID"].isin(val_vids)].copy()

print(f"\n🎬 Video-level split complete:")
print(f"   🟩 Train videos: {len(train_vids)}")
print(f"   🟦 Val videos:   {len(val_vids)}")
print(f"   🟩 Train clips:  {len(train_df)}")
print(f"   🟦 Val clips:    {len(val_df)}")

# ============================================================
# SAVE TRAIN / VAL LISTS
# ============================================================
def write_list(df, out_path):
    with open(out_path, "w") as f:
        for _, r in df.iterrows():
            clip_path = str(r["ClipPath"]).strip()
            if not os.path.isabs(clip_path):
                clip_path = os.path.join(DATASET_ROOT, clip_path)
            line = f"{clip_path} {r['Actions']}\n"
            f.write(line)
    print(f"💾 Saved {len(df)} entries → {out_path}")

write_list(train_df, TRAIN_LIST)
write_list(val_df, VAL_LIST)

# ============================================================
# SUMMARY STATS
# ============================================================
def summarize(df, name):
    counts = Counter([a for sub in df["labels_split"] for a in sub])
    total = sum(counts.values())
    print(f"\n📊 {name} set — {len(df)} clips, {len(counts)} unique actions:")
    for k, v in counts.most_common(10):
        pct = (v / total) * 100 if total > 0 else 0
        print(f"  {k:<20}: {v:>5} clips ({pct:>5.1f}%)")

summarize(train_df, "Train")
summarize(val_df, "Val")

# ============================================================
# BALANCE CHECK
# ============================================================
common_actions = set(a for sub in df["labels_split"] for a in sub)
train_actions = set(a for sub in train_df["labels_split"] for a in sub)
val_actions = set(a for sub in val_df["labels_split"] for a in sub)

missing_in_val = sorted(list(common_actions - val_actions))
if missing_in_val:
    print(f"\n⚠️ Actions missing in validation set: {missing_in_val}")
else:
    print("\n✅ All actions present in both train and val splits.")

print("\n✅ Train/Val split creation complete.")


✅ Loaded 43931 clips from /home1/jtt_1/mtp/swin-person-dataset-full/person_labels.csv

🎬 Video-level split complete:
   🟩 Train videos: 26
   🟦 Val videos:   7
   🟩 Train clips:  36690
   🟦 Val clips:    7241
💾 Saved 36690 entries → /home1/jtt_1/mtp/swin-person-dataset-full/train_list.txt
💾 Saved 7241 entries → /home1/jtt_1/mtp/swin-person-dataset-full/val_list.txt

📊 Train set — 36690 clips, 12 unique actions:
  Standing            : 16586 clips ( 28.7%)
  Walking             : 15025 clips ( 26.0%)
  Carrying            :  7248 clips ( 12.6%)
  Sitting             :  4858 clips (  8.4%)
  Pushing or Pulling  :  3545 clips (  6.1%)
  Reading             :  3336 clips (  5.8%)
  Calling             :  1726 clips (  3.0%)
  Handshaking         :  1660 clips (  2.9%)
  Running             :  1354 clips (  2.3%)
  Lying               :   913 clips (  1.6%)

📊 Val set — 7241 clips, 12 unique actions:
  Standing            :  2770 clips ( 25.0%)
  Walking             :  2612 clips ( 23.6%)
 

In [69]:
# # ============================================================
# # verify_train_val_test_csv_splits.py
# # ============================================================
# import os
# import pandas as pd
# from collections import Counter

# # ============================================================
# # CONFIGURATION
# # ============================================================
# TRAIN_ROOT = "/home1/jtt_1/mtp/swin-person-dataset-full"
# TEST_ROOT  = "/home1/jtt_1/mtp/swin-person-dataset-full-test"

# TRAIN_LIST = os.path.join(TRAIN_ROOT, "train_list.txt")
# VAL_LIST   = os.path.join(TRAIN_ROOT, "val_list.txt")
# TEST_CSV   = os.path.join(TEST_ROOT,  "person_labels.csv")  # test CSV

# # ============================================================
# # HELPER FUNCTIONS
# # ============================================================
# def get_folder_size(path):
#     total = 0
#     for dirpath, _, filenames in os.walk(path):
#         for f in filenames:
#             fp = os.path.join(dirpath, f)
#             if os.path.isfile(fp):
#                 total += os.path.getsize(fp)
#     return total

# def read_txt_split(list_path, base_root):
#     """Read train/val text lists of form: 'clip_path actions'"""
#     if not os.path.exists(list_path):
#         print(f"⚠️  Missing list file: {list_path}")
#         return []
#     pairs = []
#     with open(list_path, "r") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             parts = line.split(" ", 1)
#             clip_path = parts[0]
#             if not os.path.isabs(clip_path):
#                 clip_path = os.path.join(base_root, clip_path)
#             label = parts[1] if len(parts) > 1 else ""
#             pairs.append((os.path.normpath(clip_path), label))
#     return pairs

# def read_test_csv(csv_path, base_root):
#     """Read test CSV format person_labels.csv"""
#     if not os.path.exists(csv_path):
#         print(f"⚠️  Missing test CSV: {csv_path}")
#         return []
#     df = pd.read_csv(csv_path)
#     if "ClipPath" not in df.columns or "Actions" not in df.columns:
#         print(f"⚠️  Invalid test CSV format — missing 'ClipPath' or 'Actions'.")
#         return []
#     pairs = []
#     for _, r in df.iterrows():
#         clip_path = str(r["ClipPath"]).strip()
#         if not os.path.isabs(clip_path):
#             clip_path = os.path.join(base_root, clip_path)
#         label = str(r["Actions"])
#         pairs.append((os.path.normpath(clip_path), label))
#     return pairs

# def summarize_split(name, pairs):
#     if not pairs:
#         print(f"\n⚠️  No data for {name} split.")
#         return

#     total_files = len(pairs)
#     missing = [p for (p, _) in pairs if not os.path.exists(p)]
#     present = total_files - len(missing)

#     labels = [lbl for _, lbl in pairs if lbl.strip()]
#     all_actions = []
#     for lbl in labels:
#         all_actions.extend([a.strip() for a in lbl.split(",") if a.strip()])
#     counts = Counter(all_actions)

#     # Detect folder size
#     existing_dirs = set(os.path.dirname(p) for (p, _) in pairs if os.path.exists(p))
#     total_size = sum(get_folder_size(d) for d in existing_dirs)

#     print(f"\n📂 {name.upper()} SPLIT SUMMARY")
#     print(f"   🎞️ Total clips listed : {total_files}")
#     print(f"   ✅ Found on disk       : {present}")
#     print(f"   ❌ Missing clips       : {len(missing)}")
#     print(f"   🏷️ Unique actions      : {len(counts)}")
#     print(f"   💾 Approx folder size  : {total_size / (1024**3):.2f} GB")

#     if counts:
#         print(f"   🔝 Top 10 frequent actions:")
#         for k, v in counts.most_common(10):
#             print(f"     {k:<20}: {v}")
#     if missing:
#         print(f"   ⚠️ Missing {len(missing)} examples (e.g. {missing[:3]})")

# # ============================================================
# # MAIN EXECUTION
# # ============================================================
# print("🚀 Checking dataset splits (train / val / test CSV)...")

# splits = {
#     "train": ("txt", TRAIN_LIST, TRAIN_ROOT),
#     "val":   ("txt", VAL_LIST, TRAIN_ROOT),
#     "test":  ("csv", TEST_CSV, TEST_ROOT),
# }

# for name, (stype, path, root) in splits.items():
#     if stype == "txt":
#         pairs = read_txt_split(path, root)
#     else:
#         pairs = read_test_csv(path, root)
#     summarize_split(name, pairs)

# print("\n✅ Dataset verification complete.")


In [70]:
import os
import pandas as pd

# Folder containing the dataset files
root = "/home1/jtt_1/mtp/swin-person-dataset-full"

# Map .txt → desired manifest filenames
manifest_map = {
    "train_list.txt": "train_manifest.csv",
    "val_list.txt": "val_manifest.csv",
}

for txt_file, output_csv in manifest_map.items():
    input_path = os.path.join(root, txt_file)
    output_path = os.path.join(root, output_csv)

    if not os.path.exists(input_path):
        print(f"⚠️ File not found: {input_path}")
        continue

    rows = []
    with open(input_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Split on first space only to avoid breaking labels
            if " " in line:
                video_path, labels = line.split(" ", 1)
            else:
                continue

            # Check file exists (avoids later FileNotFoundError)
            if not os.path.exists(video_path):
                print(f"⚠️ Skipped missing: {video_path}")
                continue

            rows.append((video_path, labels.strip()))

    # Save as CSV
    df = pd.DataFrame(rows, columns=["video_path", "labels"])
    df.to_csv(output_path, index=False)
    print(f"✅ Saved manifest: {output_path} ({len(df)} valid samples)")


✅ Saved manifest: /home1/jtt_1/mtp/swin-person-dataset-full/train_manifest.csv (36690 valid samples)
✅ Saved manifest: /home1/jtt_1/mtp/swin-person-dataset-full/val_manifest.csv (7241 valid samples)


In [71]:
import pandas as pd
import os

# Root folder and manifests
root = "/home1/jtt_1/mtp/swin-person-dataset-full"
splits = ["train", "val"]

# Define label order (must match your dataset config!)
classes = [
    "Calling", "Carrying", "Drinking", "Handshaking", "Hugging",
    "Lying", "Pushing_or_Pulling", "Reading", "Running",
    "Sitting", "Standing", "Walking"
]

for split in splits:
    input_csv = os.path.join(root, f"{split}_manifest.csv")
    output_csv = os.path.join(root, f"{split}_manifest_filtered.csv")

    if not os.path.exists(input_csv):
        print(f"⚠️ Missing: {input_csv}")
        continue

    df = pd.read_csv(input_csv)
    rows = []

    for _, row in df.iterrows():
        video_path = row["video_path"].strip()
        label_str = str(row["labels"]).replace(" ", "").split(",")
        label_vec = [1 if cls in label_str else 0 for cls in classes]

        # Build one-line string like: path 0 1 0 0 ...
        line = " ".join(map(str, label_vec))
        rows.append(f"{video_path} {line}\n")

    with open(output_csv, "w") as f:
        f.writelines(rows)

    print(f"✅ Saved {output_csv} ({len(rows)} samples)")


✅ Saved /home1/jtt_1/mtp/swin-person-dataset-full/train_manifest_filtered.csv (36690 samples)
✅ Saved /home1/jtt_1/mtp/swin-person-dataset-full/val_manifest_filtered.csv (7241 samples)


In [72]:
# from decord import VideoReader
# import csv, os

# input_manifest = "/home1/jtt_1/mtp/swin-person-dataset-full/val_manifest.csv"
# output_manifest = "/home1/jtt_1/mtp/swin-person-dataset-full/val_manifest_filtered.csv"

# os.makedirs(os.path.dirname(output_manifest), exist_ok=True)

# with open(input_manifest, newline='') as f_in, open(output_manifest, "w", newline='') as f_out:
#     reader = csv.reader(f_in)
#     writer = csv.writer(f_out)
    
#     header = next(reader, None)
#     if header:
#         writer.writerow(header)
    
#     for row in reader:
#         if not row or len(row) < 1:
#             continue
#         path = row[0].strip()
#         if not os.path.exists(path):
#             print(f"⚠️ File missing: {path}")
#             continue
#         try:
#             vr = VideoReader(path)
#             if len(vr) >= 5:  # require at least 5 frames
#                 writer.writerow(row)
#         except Exception as e:
#             print(f"⚠️ Skipped {path}: {e}")

# print(f"✅ Saved filtered manifest → {output_manifest}")


In [73]:
# from decord import VideoReader
# import csv, os

# input_manifest = "/home1/jtt_1/mtp/swin-person-dataset-full/train_manifest.csv"
# output_manifest = "/home1/jtt_1/mtp/swin-person-dataset-full/train_manifest_filtered.csv"

# os.makedirs(os.path.dirname(output_manifest), exist_ok=True)

# with open(input_manifest, newline='') as f_in, open(output_manifest, "w", newline='') as f_out:
#     reader = csv.reader(f_in)
#     writer = csv.writer(f_out)
    
#     header = next(reader, None)
#     if header:
#         writer.writerow(header)
    
#     for row in reader:
#         if not row or len(row) < 1:
#             continue
#         path = row[0].strip()
#         if not os.path.exists(path):
#             print(f"⚠️ File missing: {path}")
#             continue
#         try:
#             vr = VideoReader(path)
#             if len(vr) >= 5:  # require at least 5 frames
#                 writer.writerow(row)
#         except Exception as e:
#             print(f"⚠️ Skipped {path}: {e}")

# print(f"✅ Saved filtered manifest → {output_manifest}")


#### new-former

In [74]:
# !python tools/train.py configs/recognition/swin/swin_tiny_patch244_window877_okutama_multilabel.py \
#   --work-dir /home1/jtt_1/mtp/swin-person-dataset-full/swin-tiny-multilabel-output


In [75]:
# python tools/test.py configs/recognition/swin/swin_tiny_patch244_window877_okutama_multilabel.py \
#   /home1/jtt_1/mtp/swin-person-dataset-full/swin-tiny-multilabel-output/latest.pth \
#   --eval mean_average_precision
